# TEMPLATE UTS DM II — CLUSTERING TEKS (Google Colab)

**Cara tercepat:** jalankan **Runtime → Run all** dengan `SUMBER = "contoh"` terlebih dahulu. Pada UTS, ganti `SUMBER = "upload"`, pilih CSV/XLSX/JSON/TXT, lalu cek nama kolom teks. Tidak perlu menulis ulang tahapan lainnya.

**Alur:** upload → cek data → sampling → cleaning → Sastrawi/TextBlob → WordCloud/frekuensi → TF-IDF → Elbow & Silhouette → K-Means → SVD 2D → kata dominan + contoh kalimat.

- Dataset besar? Ubah `N_SAMPEL = 100`, `300`, atau `1000` (stemming Sastrawi lebih lambat). Sampling dilakukan **sebelum stemming**.
- Ini **unsupervised clustering**. Kolom label asli bila ada hanya dipakai sebagai informasi pembanding.
- Selalu baca penjelasan **di bawah tiap sel kode**. Semua sel kode sengaja diberi `print`, `display`, atau grafik sehingga menghasilkan output.
- **Tidak ada `def` maupun `try-except`** dalam kode latihan.

**Keyword cepat:** `pandas read csv encoding`, `Sastrawi stemming Indonesian`, `TextBlob tokenize`, `sklearn TfidfVectorizer`, `KMeans text silhouette_score`, `TruncatedSVD text clustering`.

## 0. KONFIGURASI — hanya ganti bagian ini

In [ ]:
# [GANTI SAAT UTS]
SUMBER = "contoh"         # "contoh", "upload", atau "path"
PATH_DATA = "/content/dataset.csv"  # dipakai hanya jika SUMBER="path" (mis. file di Google Drive)
KOLOM_TEKS = "otomatis"    # "otomatis" atau nama kolom, mis. "Tweet"
KOLOM_LABEL = ""          # opsional, contoh: "Sentiment"; bukan fitur clustering
SEPARATOR = ","           # untuk CSV, bisa diganti ";"
ENCODING = "utf-8"        # untuk CSV/TXT; coba "latin-1" jika encoding bermasalah
JSON_LINES = False        # True untuk berkas .jsonl / JSON lines
N_SAMPEL = 120            # misalnya 100, 300, atau 1000
METODE = "sastrawi"       # "sastrawi", "textblob", atau "simple"
BAHASA = "id"            # "id" Bahasa Indonesia, "en" English; Sastrawi khusus "id"
K_FINAL = 3              # jumlah cluster final, sesuaikan grafik
RANDOM_STATE = 42
print("SUMBER:", SUMBER, "| sampel maksimum:", N_SAMPEL, "| metode:", METODE, "| k:", K_FINAL)

**Sesudah menjalankan cell — petunjuk & interpretasi:** Ubah **SUMBER**, **KOLOM_TEKS**, **N_SAMPEL**, **METODE**, dan **K_FINAL**. Untuk file di Google Drive, mount Drive dahulu di sel terpisah bila perlu, lalu isi `PATH_DATA`. `METODE="sastrawi"` hanya untuk bahasa Indonesia. **Keyword:** `google colab upload csv`, `pandas read_csv sep encoding`.

## 1. Install library

In [ ]:
# Instalasi diperlukan sekali per runtime Colab; abaikan warning update pip
%pip -q install Sastrawi textblob wordcloud openpyxl
print("Instalasi library selesai.")

**Sesudah menjalankan cell — petunjuk & interpretasi:** `%pip` menginstal library pada kernel notebook. Jika runtime baru dibuat, jalankan lagi. **Keyword:** `pip install Sastrawi Google Colab`.

## 2. Import library + referensi pandas

In [ ]:
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from wordcloud import WordCloud
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import TruncatedSVD

# library utama: pandas, matplotlib, sklearn; "from" mengambil objek tertentu dari library
print("Baca CSV: pd.read_csv('file.csv')")
print("Baca Excel: pd.read_excel('file.xlsx')")
print("Baca JSON: pd.read_json('file.json')")
print("Baca teks TXT: open('file.txt', encoding='utf-8').read()")
print("Baca Parquet: pd.read_parquet('file.parquet')")

**Sesudah menjalankan cell — petunjuk & interpretasi:** `pandas` untuk tabel, `re` untuk regular expression, `TextBlob` untuk tokenisasi berbasis bahasa (tidak setara dengan stemmer Bahasa Indonesia), `Sastrawi` untuk stemming Indonesia. `sklearn` untuk TF-IDF, K-Means, dan skor cluster. **Keyword:** `from sklearn.cluster import KMeans`.

## 3. Baca data — contoh / upload / path

In [ ]:
# DATA CONTOH bisa langsung dijalankan tanpa unggah file
if SUMBER == "contoh":
    topik = {
        "kuliah": ["mahasiswa belajar data mining di kampus", "dosen menjelaskan materi dan tugas kuliah", "kelas praktikum python membahas data", "mahasiswa menyiapkan ujian dan belajar statistik", "perkuliahan membahas algoritma pembelajaran mesin", "tugas kelompok mahasiswa selesai di kelas"],
        "makanan": ["restoran ini menjual makanan yang enak", "rasa nasi goreng sangat lezat", "pelayan restoran ramah dan makanannya segar", "minuman dan makanan di kantin murah", "saya membeli roti untuk sarapan pagi", "masakan ayam dan sayur terasa nikmat"],
        "olahraga": ["tim sepak bola menang dalam pertandingan", "pemain latihan lari di lapangan", "pertandingan basket berlangsung sangat seru", "olahraga rutin menyehatkan badan", "pelatih memberikan strategi kepada pemain", "penggemar menonton final sepak bola"]
    }
    df = pd.DataFrame([(kalimat + " hari ini", topik_) for topik_, daftar in topik.items() for kalimat in daftar] * 3,
                      columns=["teks", "topik_referensi"])
    KOLOM_TEKS = "teks"
    KOLOM_LABEL = "topik_referensi"
else:
    if SUMBER == "upload":
        from google.colab import files
        hasil_upload = files.upload()
        nama_file = next(iter(hasil_upload))
    else:
        nama_file = PATH_DATA
    nama = nama_file.lower()
    if nama.endswith(".csv"):
        df = pd.read_csv(nama_file, sep=SEPARATOR, encoding=ENCODING)
    elif nama.endswith((".xlsx", ".xls")):
        df = pd.read_excel(nama_file)
    elif nama.endswith((".json", ".jsonl")):
        df = pd.read_json(nama_file, lines=JSON_LINES or nama.endswith(".jsonl"))
    elif nama.endswith(".txt"):
        with open(nama_file, encoding=ENCODING) as f:
            df = pd.DataFrame({"teks": [baris.strip() for baris in f if baris.strip()]})
        KOLOM_TEKS = "teks"
    else:
        raise ValueError("Gunakan CSV/XLSX/JSON/JSONL/TXT.")
print("Shape awal:", df.shape)
print("Kolom tersedia:", df.columns.tolist())
display(df.head(5))

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Cara masukkan data:** set `SUMBER="upload"` → jalankan → pilih file. Untuk file TXT, **satu baris = satu dokumen**. CSV harus memiliki kolom berisi teks, mis. `ulasan`, `text`, `Tweet`. Jika `.csv` terpisah dengan titik koma, ganti `SEPARATOR`. **Keyword:** `pandas read_excel column names`, `google colab files.upload`.

## 4. Pilih kolom teks, hilangkan NA/duplikat, kemudian SAMPLING

In [ ]:
# Jika belum tahu nama kolom, sistem mencoba nama umum lalu kolom string pertama
if KOLOM_TEKS == "otomatis":
    pilihan = [c for c in df.columns if str(c).lower() in ["text", "teks", "tweet", "corpus", "review", "ulasan", "komentar", "content", "sentence"]]
    KOLOM_TEKS = pilihan[0] if pilihan else df.select_dtypes(include=["object", "string"]).columns[0]
assert KOLOM_TEKS in df.columns, "Kolom teks tidak ada; cek df.columns dan ubah KOLOM_TEKS."

df = df.dropna(subset=[KOLOM_TEKS]).copy()
df[KOLOM_TEKS] = df[KOLOM_TEKS].astype(str).str.strip()
df = df[df[KOLOM_TEKS] != ""].drop_duplicates(subset=[KOLOM_TEKS])
jumlah_bersih = len(df)
df = df.sample(n=min(N_SAMPEL, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)
assert len(df) >= 3, "Butuh minimal 3 dokumen berbeda untuk demo pemilihan k."
print("Kolom teks:", KOLOM_TEKS)
print("Sesudah hapus kosong/duplikat:", jumlah_bersih)
print("Sesudah sampling:", df.shape)
display(df[[KOLOM_TEKS]].head())

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Sampling:** `df.sample(n=min(1000,len(df)),random_state=42)` memilih baris acak tanpa pengembalian; jumlah sampel tidak melebihi data. Jika label kategori harus seimbang, pertimbangkan stratified sampling (untuk eksplorasi, bukan memakai label sebagai fitur). **Keyword:** `pandas sample n random_state`, `drop_duplicates subset`.

## 5. Case folding, cleaning, stopwords dan perbandingan TextBlob–Sastrawi

In [ ]:
# Cleaning sederhana (tanpa fungsi def)
df["clean"] = df[KOLOM_TEKS].str.lower()
df["clean"] = df["clean"].str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
df["clean"] = df["clean"].str.replace(r"[^a-zA-Z\s]", " ", regex=True)
df["clean"] = df["clean"].str.replace(r"\s+", " ", regex=True).str.strip()

if BAHASA == "id":
    # stopword sederhana yg sama untuk semua metode agar perbandingan adil
    stop_kata = set("yang dan di ke dari ini itu untuk saya kamu dia kita dengan pada adalah juga sangat sebuah atau dalam tersebut telah sudah bisa akan tidak karena agar sebagai oleh para pun kepada serta ada saat hari yg gak ga nya aja banget sih".split())
else:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    stop_kata = set(ENGLISH_STOP_WORDS)

# Tokenisasi sederhana: split; TextBlob.words memakai tokenizer dari NLTK
if METODE == "textblob":
    import nltk
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
    df["tokens"] = df["clean"].apply(lambda s: list(TextBlob(s).words))
else:
    df["tokens"] = df["clean"].str.split()

df["bersih"] = df["tokens"].apply(lambda a: " ".join(w for w in a if w not in stop_kata and len(w)>1))
if METODE == "sastrawi":
    assert BAHASA == "id", "Sastrawi hanya cocok untuk bahasa Indonesia; ganti METODE atau BAHASA."
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
    df["bersih"] = df["bersih"].apply(stemmer.stem)

df = df[df["bersih"].str.len() > 0].reset_index(drop=True)
print("Metode:", METODE, "| dokumen tersedia:", len(df))
display(df[[KOLOM_TEKS, "clean", "bersih"]].head(5))

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Perbedaan:** Sastrawi menghapus imbuhan (`makanan`→`makan` dalam konteks tertentu) sedangkan TextBlob terutama menyediakan tokenisasi/analisis teks dan tidak otomatis melakukan stemming Bahasa Indonesia. Stopword disaring pada kedua metode. Cleaning di atas membuang angka dan karakter non-A–Z; ubah regex jika nomor/tanda baca penting. **Keyword:** `Sastrawi StemmerFactory create_stemmer`, `TextBlob words nltk punkt_tab`.

## 6. WordCloud + kata paling sering

In [ ]:
kata = pd.Series(" ".join(df["bersih"]).split())
freq = kata.value_counts().head(15)
print("15 kata terbanyak:")
display(freq.to_frame("frekuensi"))
plt.figure(figsize=(9, 3))
freq.sort_values().plot(kind="barh", title="15 kata paling sering")
plt.xlabel("Frekuensi")
plt.tight_layout(); plt.show()
plt.figure(figsize=(9, 4))
plt.imshow(WordCloud(width=900, height=400, background_color="white").generate(" ".join(df["bersih"])))
plt.axis("off"); plt.title("WordCloud"); plt.show()

**Sesudah menjalankan cell — petunjuk & interpretasi:** Kata yang paling sering muncul menunjukkan kosakata dominan, **bukan otomatis tema cluster**. Jika kata umum mendominasi, perbaiki stopword. **Keyword:** `pandas value_counts word frequency barplot`, `WordCloud python`.

## 7. TF-IDF: ubah dokumen menjadi matriks angka

In [ ]:
vectorizer = TfidfVectorizer(max_features=1500, min_df=1)
X = vectorizer.fit_transform(df["bersih"])
print("Shape TF-IDF (dokumen, fitur kata):", X.shape)
print("Jumlah kata unik terpakai:", len(vectorizer.get_feature_names_out()))
display(pd.DataFrame(X[:5].toarray(), columns=vectorizer.get_feature_names_out()).iloc[:, :12].round(2))

**Sesudah menjalankan cell — petunjuk & interpretasi:** `X.shape = (n_dokumen, n_kata)`; K-Means **tidak bisa menerima teks mentah**. Untuk data kecil jangan langsung menggunakan `min_df=5` karena semua fitur dapat hilang. **Keyword:** `TfidfVectorizer fit_transform shape`, `ValueError empty vocabulary`.

## 8. Elbow + Silhouette untuk memilih k

In [ ]:
# Hanya kandidat k yang masuk akal untuk banyaknya dokumen unik
n_unik = len(set(df["bersih"]))
assert n_unik >= 3, "Kurang dari 3 dokumen berbeda setelah preprocessing; gunakan data lebih bervariasi."
k_candidates = list(range(2, min(7, len(df)-1, n_unik) + 1))
inertia, sil = [], []
for k in k_candidates:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lab = km.fit_predict(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, lab) if 1 < len(set(lab)) < len(df) else np.nan)
hasil_k = pd.DataFrame({"k":k_candidates, "inertia":inertia, "silhouette":sil})
display(hasil_k.round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(k_candidates, inertia, "o-"); ax[0].set(title="Elbow", xlabel="k", ylabel="Inertia")
ax[1].plot(k_candidates, sil, "o-"); ax[1].set(title="Silhouette", xlabel="k", ylabel="Skor")
plt.tight_layout(); plt.show()

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Elbow:** pilih titik ketika penurunan inertia mulai melandai. **Silhouette:** semakin mendekati 1 cenderung lebih terpisah (sekitar 0 tumpang tindih, negatif mengindikasikan kemungkinan kurang tepat). Gunakan pertimbangan keduanya, bukan otomatis jumlah kelas. **Keyword:** `KMeans elbow method`, `silhouette_score clustering`.

## 9. K-Means final + evaluasi

In [ ]:
K = min(K_FINAL, len(df)-1, n_unik)
model = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = model.fit_predict(X)
print("k dipakai:", K, "| klaster aktual:", df["cluster"].nunique())
print("Ukuran cluster:")
display(df["cluster"].value_counts().sort_index().to_frame("jumlah"))
if 1 < df["cluster"].nunique() < len(df):
    print("Silhouette:", round(silhouette_score(X, df["cluster"]), 4))
    print("Davies-Bouldin:", round(davies_bouldin_score(X.toarray(), df["cluster"]), 4))
else:
    print("Evaluasi tidak terdefinisi karena jumlah cluster berbeda terlalu sedikit/banyak.")

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Silhouette lebih tinggi** dan **Davies–Bouldin lebih rendah** biasanya mengindikasikan pemisahan cluster lebih baik, tetapi tidak menjamin cluster memiliki makna tematik. Label 0,1,2 hanyalah indeks cluster, **bukan** kategori sentimen. **Keyword:** `sklearn KMeans fit_predict`, `davies_bouldin_score`.

## 10. Visualisasi 2D menggunakan TruncatedSVD

In [ ]:
# SVD cocok untuk TF-IDF sparse; ini visualisasi 2D, bukan hasil fitur asli
if X.shape[1] >= 2:
    coords = TruncatedSVD(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
else:
    coords = np.column_stack([X.toarray()[:, 0], np.zeros(X.shape[0])])
print("Shape proyeksi:", coords.shape)
plt.figure(figsize=(7, 5))
plt.scatter(coords[:, 0], coords[:, 1], c=df["cluster"], cmap="tab10", s=35)
plt.xlabel("SVD 1"); plt.ylabel("SVD 2"); plt.title("K-Means teks (proyeksi 2D)")
plt.tight_layout(); plt.show()

**Sesudah menjalankan cell — petunjuk & interpretasi:** Plot 2D menyederhanakan ruang TF-IDF berdimensi banyak. Titik yang saling berdekatan pada plot belum tentu selalu dekat pada seluruh dimensi asal. **Keyword:** `TruncatedSVD sparse TFIDF scatter plot`.

## 11. Interpretasi keyword + contoh dokumen tiap cluster

In [ ]:
terms = vectorizer.get_feature_names_out()
for c in range(K):
    idx = np.argsort(model.cluster_centers_[c])[-8:][::-1]
    print("\nCLUSTER", c, "|", len(df[df.cluster == c]), "dokumen")
    print("Kata paling berbobot:", ", ".join(terms[idx]))
    print("Contoh teks:")
    for kalimat in df.loc[df.cluster == c, KOLOM_TEKS].head(3):
        print("-", kalimat[:180])
if KOLOM_LABEL and KOLOM_LABEL in df.columns:
    print("\nTabulasi terhadap label referensi (opsional):")
    display(pd.crosstab(df["cluster"], df[KOLOM_LABEL]))
print("\nIngat: beri nama cluster berdasarkan isi dokumen, bukan angka clusternya.")

**Sesudah menjalankan cell — petunjuk & interpretasi:** Keyword diambil dari **bobot centroid TF-IDF**, bukan frekuensi absolut. Cek beberapa dokumen sebelum menamai cluster. Jika punya label referensi, `pd.crosstab` untuk melihat kecocokan **setelah** clustering. **Keyword:** `KMeans cluster centers top tfidf terms`, `pandas crosstab`.

## 12. Simpan hasil clustering

In [ ]:
df.to_csv("hasil_cluster_teks.csv", index=False)
print("Disimpan: hasil_cluster_teks.csv |", df.shape)
display(df[[KOLOM_TEKS, "bersih", "cluster"]].head())

**Sesudah menjalankan cell — petunjuk & interpretasi:** Unduh dari panel **Files** Colab atau gunakan `from google.colab import files; files.download("hasil_cluster_teks.csv")`. **Keyword:** `pandas to_csv index false`, `colab download csv`.

## Checklist UTS — Text

1. Cek nama kolom dan beberapa teks asli. 2. Bersihkan kosong/duplikat. 3. Sampling **sebelum stemming**. 4. Cleaning + Sastrawi/TextBlob. 5. WordCloud/barplot. 6. TF-IDF. 7. Elbow & Silhouette. 8. K-Means. 9. Interpretasi keyword, contoh dokumen, ukuran cluster. 10. Simpan CSV.

**Jika soal khusus TextBlob vs Sastrawi:** jalankan pipeline dengan `METODE="sastrawi"`, catat hasil, lalu ulangi dengan `METODE="textblob"`. Keduanya *tidak* punya fungsi linguistik identik untuk Bahasa Indonesia; jelaskan perbedaan preprocessing, bukan hanya skornya.